This program performs the following tasks:
1. Imports the required packages, dependencies and libraries
2. Initializes the test case dataset
3. Runs tasks to evaluate each pLM/model (through "adapters")
4. Returns the error and AURPC of each model
# Step 1: Initialize required packages and datasets

## 1.1 Prepare the envs and dependencies overall

In [1]:
import os
import subprocess
import sys

print("Setting up isolated environments...")
os.makedirs("./envs", exist_ok=True)
os.makedirs("evaluation_outputs", exist_ok=True)

base_pkgs = [
    "python=3.10", "pytorch", "torchvision", "torchaudio", 
    "transformers", "pandas", "numpy", "tqdm", "cd-hit", "hmmer"
]
conda_args = ["-c", "pytorch", "-c", "conda-forge", "-c", "bioconda", "-y"]

def setup_conda_env(env_name, extra_pkgs=None):
    env_path = os.path.abspath(f"./envs/{env_name}")
    python_exec = os.path.join(env_path, "bin", "python")
    pip_exec = os.path.join(env_path, "bin", "pip")
    
    if not os.path.exists(python_exec):
        print(f"Creating {env_name} environment...")
        pkgs = base_pkgs + (extra_pkgs if extra_pkgs else [])
        subprocess.run(["conda", "create", "--prefix", env_path] + pkgs + conda_args, check=True)
        subprocess.run([pip_exec, "install", "abnumber"], check=True)
    else:
        print(f"{env_name} environment already exists.")
    return env_path, python_exec

# 1. ESM-2
env_esm, esm_python = setup_conda_env("plm_esm")
# 2. ProtT5
env_prott5, prott5_python = setup_conda_env("plm_prott5", ["sentencepiece", "protobuf"])
# 3. Ankh
env_ankh, ankh_python = setup_conda_env("plm_ankh", ["sentencepiece", "protobuf", "ankh"])

# CRITICAL: Force PyTorch >= 2.6 across ALL environments to satisfy security requirements
print("\nEnforcing PyTorch >= 2.6 across all environments...")
for env_name in ["plm_esm", "plm_prott5", "plm_ankh"]:
    pip_path = os.path.abspath(f"./envs/{env_name}/bin/pip")
    subprocess.run([pip_path, "install", "--upgrade", "torch", "torchvision", "torchaudio"], check=True)

print("Installing main notebook evaluation dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "tabulate", "datasets", "scikit-learn", "abnumber"], check=True)
print("\nAll environments, dependencies, and security patches are ready!")

Setting up isolated environments...
plm_esm environment already exists.
plm_prott5 environment already exists.
plm_ankh environment already exists.

Enforcing PyTorch >= 2.6 across all environments...
Installing main notebook evaluation dependencies...

All environments, dependencies, and security patches are ready!


# 1.2 Prepare dataset

In [2]:
import os
import subprocess
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

output_dir = "evaluation_outputs"

print("Downloading dataset from Hugging Face...")
dataset = load_dataset("AbBibench/Antibody_Binding_Benchmark_Dataset", split="train")
df = dataset.to_pandas().dropna(subset=['heavy_chain_seq', 'binding_score'])
df['sequence'] = df['heavy_chain_seq']

# 1. Lab-realistic Threshold (Top 5% are binders)
threshold = df['binding_score'].quantile(0.95)
df['label'] = (df['binding_score'] >= threshold).astype(int)

# 2. CD-HIT Clustering to prevent data leakage (80% identity)
fasta_path = os.path.join(output_dir, "sequences.fasta")
with open(fasta_path, "w") as f:
    for i, seq in enumerate(df['sequence']):
        f.write(f">seq_{i}\n{seq}\n")

cdhit_out = os.path.join(output_dir, "clustered")
subprocess.run([
    os.path.abspath("./envs/plm_esm/bin/cd-hit"),
    "-i", fasta_path, "-o", cdhit_out, 
    "-c", "0.8", "-n", "5", "-M", "0"
], check=True, stdout=subprocess.DEVNULL)

cluster_dict = {}
with open(cdhit_out + ".clstr", "r") as f:
    current_cluster = None
    for line in f:
        if line.startswith(">Cluster"):
            current_cluster = int(line.split()[1])
        else:
            seq_idx = int(line.split(">")[1].split("...")[0].split("_")[1])
            cluster_dict[seq_idx] = current_cluster

df['cluster_id'] = df.index.map(cluster_dict)

gss = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['cluster_id']))

df['split'] = 'train'
df.iloc[test_idx, df.columns.get_loc('split')] = 'test'

final_df = df[['sequence', 'label', 'split', 'cluster_id']].reset_index(drop=True)
final_df.to_csv(os.path.join(output_dir, "abag_full_dataset.csv"), index=False)
print(f"Dataset ready. True binder percentage: {final_df['label'].mean() * 100:.2f}%")

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Dataset ready. True binder percentage: 5.00%


# Step 2: Run the models
## Step 2.1: ESM-2

In [3]:
import os
import subprocess

esm_script = """import os, torch, pickle, gc
import pandas as pd
from transformers import AutoTokenizer, EsmModel
from tqdm import tqdm
from abnumber import Chain

def get_cdr_indices(sequence, scheme='imgt'):
    try:
        chain = Chain(sequence, scheme=scheme)
        c1, c2, c3 = sequence.find(chain.cdr1_seq), sequence.find(chain.cdr2_seq), sequence.find(chain.cdr3_seq)
        if -1 in (c1, c2, c3): return None
        return {'CDR1': (c1, c1 + len(chain.cdr1_seq)), 'CDR2': (c2, c2 + len(chain.cdr2_seq)), 'CDR3': (c3, c3 + len(chain.cdr3_seq))}
    except Exception: return None

df = pd.read_csv("evaluation_outputs/abag_full_dataset.csv")
sequences, labels = df["sequence"].tolist(), df["label"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D").to(device)
model.eval()

embeddings = []
TOKEN_OFFSET = 1  # ESM-2 uses <cls> token offset

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), 64), desc="ESM-2 CDR Extraction"):
        batch_seqs = sequences[i:i+64]
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        outputs = model(**inputs)
        
        for j, seq in enumerate(batch_seqs):
            last_hidden = outputs.last_hidden_state[j]
            cdr_bounds = get_cdr_indices(seq)
            if cdr_bounds:
                cdrs = torch.cat([
                    last_hidden[cdr_bounds['CDR1'][0] + TOKEN_OFFSET : cdr_bounds['CDR1'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR2'][0] + TOKEN_OFFSET : cdr_bounds['CDR2'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR3'][0] + TOKEN_OFFSET : cdr_bounds['CDR3'][1] + TOKEN_OFFSET]
                ], dim=0)
                embeddings.append(cdrs.mean(dim=0).cpu().numpy())
            else:
                embeddings.append(last_hidden[TOKEN_OFFSET : min(len(seq), 510) + TOKEN_OFFSET].mean(dim=0).cpu().numpy())

with open("evaluation_outputs/esm2_abag_full_results.pkl", "wb") as f:
    pickle.dump({"embeddings": embeddings, "labels": labels}, f)
"""

with open("run_esm.py", "w") as f: f.write(esm_script)
subprocess.run([os.path.abspath("./envs/plm_esm/bin/python"), "run_esm.py"], check=True)
print("ESM-2 extraction complete.")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5670.38it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
ESM-2 CDR Extraction: 100%|██████████| 3371/3371 [03:19<00:00, 16.87it/s]


ESM-2 extraction complete.


## Step 2.1.1 ESM2 3B

In [ ]:
import os
import subprocess

esm3b_script = """import os, torch, pickle, gc
import pandas as pd
from transformers import AutoTokenizer, EsmModel
from tqdm import tqdm
from abnumber import Chain

def get_cdr_indices(sequence, scheme='imgt'):
    try:
        chain = Chain(sequence, scheme=scheme)
        c1, c2, c3 = sequence.find(chain.cdr1_seq), sequence.find(chain.cdr2_seq), sequence.find(chain.cdr3_seq)
        if -1 in (c1, c2, c3): return None
        return {'CDR1': (c1, c1 + len(chain.cdr1_seq)), 'CDR2': (c2, c2 + len(chain.cdr2_seq)), 'CDR3': (c3, c3 + len(chain.cdr3_seq))}
    except Exception: return None

df = pd.read_csv("evaluation_outputs/abag_full_dataset.csv")
sequences, labels = df["sequence"].tolist(), df["label"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "facebook/esm2_t36_3B_UR50D"

print(f"Loading {model_name} in float16 for 20GB VRAM optimization...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

embeddings = []
TOKEN_OFFSET = 1  # ESM-2 class token offset
batch_size = 2  # Optimized safe batch size for 3B model on 20GB VRAM

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), batch_size), desc="ESM-2 (3B) CDR Extraction"):
        batch_seqs = sequences[i:i+batch_size]
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        outputs = model(**inputs)
        
        for j, seq in enumerate(batch_seqs):
            last_hidden = outputs.last_hidden_state[j]
            cdr_bounds = get_cdr_indices(seq)
            if cdr_bounds:
                cdrs = torch.cat([
                    last_hidden[cdr_bounds['CDR1'][0] + TOKEN_OFFSET : cdr_bounds['CDR1'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR2'][0] + TOKEN_OFFSET : cdr_bounds['CDR2'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR3'][0] + TOKEN_OFFSET : cdr_bounds['CDR3'][1] + TOKEN_OFFSET]
                ], dim=0)
                embeddings.append(cdrs.mean(dim=0).cpu().float().numpy())
            else:
                embeddings.append(last_hidden[TOKEN_OFFSET : min(len(seq), 510) + TOKEN_OFFSET].mean(dim=0).cpu().float().numpy())

with open("evaluation_outputs/esm2_3b_abag_full_results.pkl", "wb") as f:
    pickle.dump({"embeddings": embeddings, "labels": labels}, f)

del model, tokenizer, embeddings; gc.collect(); torch.cuda.empty_cache()
"""

with open("run_esm3b.py", "w") as f: f.write(esm3b_script)
subprocess.run([os.path.abspath("./envs/plm_esm/bin/python"), "run_esm3b.py"], check=True)
print("ESM-2 (3B) extraction complete.")

## Step 2.2: ProtT5

In [4]:
import os
import subprocess

prott5_script = """import os, torch, pickle, re, gc
import pandas as pd
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm
from abnumber import Chain

def get_cdr_indices(sequence, scheme='imgt'):
    try:
        chain = Chain(sequence, scheme=scheme)
        c1, c2, c3 = sequence.find(chain.cdr1_seq), sequence.find(chain.cdr2_seq), sequence.find(chain.cdr3_seq)
        if -1 in (c1, c2, c3): return None
        return {'CDR1': (c1, c1 + len(chain.cdr1_seq)), 'CDR2': (c2, c2 + len(chain.cdr2_seq)), 'CDR3': (c3, c3 + len(chain.cdr3_seq))}
    except Exception: return None

df = pd.read_csv("evaluation_outputs/abag_full_dataset.csv")
sequences, labels = df["sequence"].tolist(), df["label"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc", do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc").to(device)
model.eval()
if torch.cuda.is_available(): model.half()

embeddings = []
TOKEN_OFFSET = 0

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), 16), desc="ProtT5 CDR Extraction"):
        batch_seqs_raw = sequences[i:i+16]
        batch_spaced = [" ".join(list(re.sub(r"[UZOB]", "X", seq))) for seq in batch_seqs_raw]
        inputs = tokenizer(batch_spaced, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        outputs = model(**inputs)
        
        for j, seq in enumerate(batch_seqs_raw):
            last_hidden = outputs.last_hidden_state[j]
            cdr_bounds = get_cdr_indices(seq)
            if cdr_bounds:
                cdrs = torch.cat([
                    last_hidden[cdr_bounds['CDR1'][0] + TOKEN_OFFSET : cdr_bounds['CDR1'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR2'][0] + TOKEN_OFFSET : cdr_bounds['CDR2'][1] + TOKEN_OFFSET],
                    last_hidden[cdr_bounds['CDR3'][0] + TOKEN_OFFSET : cdr_bounds['CDR3'][1] + TOKEN_OFFSET]
                ], dim=0)
                embeddings.append(cdrs.mean(dim=0).cpu().float().numpy())
            else:
                embeddings.append(last_hidden[TOKEN_OFFSET : min(len(seq), 512) + TOKEN_OFFSET].mean(dim=0).cpu().float().numpy())

with open("evaluation_outputs/prott5_abag_full_results.pkl", "wb") as f:
    pickle.dump({"embeddings": embeddings, "labels": labels}, f)
"""

with open("run_prott5.py", "w") as f: f.write(prott5_script)
subprocess.run([os.path.abspath("./envs/plm_prott5/bin/python"), "run_prott5.py"], check=True)
print("ProtT5 extraction complete.")

ProtT5 CDR Extraction: 100%|██████████| 13482/13482 [40:09<00:00,  5.60it/s]


ProtT5 extraction complete.


## PLM Akinah

In [9]:
import os
import subprocess

ankh_fixed_script = """import os, torch, pickle, gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from abnumber import Chain

def get_cdr_indices(sequence, scheme='imgt'):
    try:
        chain = Chain(sequence, scheme=scheme)
        c1, c2, c3 = sequence.find(chain.cdr1_seq), sequence.find(chain.cdr2_seq), sequence.find(chain.cdr3_seq)
        if -1 in (c1, c2, c3): return None
        return {'CDR1': (c1, c1 + len(chain.cdr1_seq)), 'CDR2': (c2, c2 + len(chain.cdr2_seq)), 'CDR3': (c3, c3 + len(chain.cdr3_seq))}
    except Exception: return None

df = pd.read_csv("evaluation_outputs/abag_full_dataset.csv")
sequences, labels = df["sequence"].tolist(), df["label"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("ElnaggarLab/ankh-base")
model = AutoModelForSeq2SeqLM.from_pretrained("ElnaggarLab/ankh-base").to(device)
model.eval()

embeddings = []
# Ankh character tokenizer alignment check
TOKEN_OFFSET = 0 

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), 4), desc="Ankh Fixed CDR Extraction"):
        batch_seqs = sequences[i:i+4]
        batch_list = [list(seq) for seq in batch_seqs]
        inputs = tokenizer(batch_list, add_special_tokens=True, padding=True, is_split_into_words=True, return_tensors="pt").to(device)
        
        encoder_outputs = model.get_encoder()(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        last_hidden = encoder_outputs.last_hidden_state
        
        for j, seq in enumerate(batch_seqs):
            seq_hidden = last_hidden[j]
            cdr_bounds = get_cdr_indices(seq)
            
            if cdr_bounds:
                # Ensure indices don't exceed sequence length tensor bounds
                max_len = seq_hidden.size(0)
                c1 = (min(cdr_bounds['CDR1'][0], max_len), min(cdr_bounds['CDR1'][1], max_len))
                c2 = (min(cdr_bounds['CDR2'][0], max_len), min(cdr_bounds['CDR2'][1], max_len))
                c3 = (min(cdr_bounds['CDR3'][0], max_len), min(cdr_bounds['CDR3'][1], max_len))
                
                cdrs = torch.cat([
                    seq_hidden[c1[0]:c1[1]],
                    seq_hidden[c2[0]:c2[1]],
                    seq_hidden[c3[0]:c3[1]]
                ], dim=0)
                if cdrs.numel() > 0:
                    embeddings.append(cdrs.mean(dim=0).cpu().numpy())
                    continue
            
            seq_len = min(len(seq), seq_hidden.size(0))
            embeddings.append(seq_hidden[TOKEN_OFFSET : seq_len + TOKEN_OFFSET].mean(dim=0).cpu().numpy())

with open("evaluation_outputs/ankh_abag_full_results.pkl", "wb") as f:
    pickle.dump({"embeddings": embeddings, "labels": labels}, f)

del model, tokenizer, embeddings; gc.collect(); torch.cuda.empty_cache()
"""

with open("run_ankh_fixed.py", "w") as f: f.write(ankh_fixed_script)
subprocess.run([os.path.abspath("./envs/plm_ankh/bin/python"), "run_ankh_fixed.py"], check=True)
print("Ankh extraction complete.")

Ankh Fixed CDR Extraction:   8%|▊         | 4132/53925 [05:07<1:01:51, 13.42it/s]
Traceback (most recent call last):
  File "/mnt/natanon/EvalTests/run_ankh_fixed.py", line 33, in <module>
    encoder_outputs = model.get_encoder()(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
  File "/mnt/natanon/EvalTests/envs/plm_ankh/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1778, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/mnt/natanon/EvalTests/envs/plm_ankh/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1789, in _call_impl
    return forward_call(*args, **kwargs)
  File "/mnt/natanon/EvalTests/envs/plm_ankh/lib/python3.10/site-packages/transformers/models/t5/modeling_t5.py", line 1100, in forward
    layer_outputs = layer_module(
  File "/mnt/natanon/EvalTests/envs/plm_ankh/lib/python3.10/site-packages/transformers/modeling_layers.py", line 94, in __call__
    return super().__call__(*args, **kwargs)
  Fi

KeyboardInterrupt: 

# Step 3: Evaluate the model results against known answers

In [8]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score

output_dir = "evaluation_outputs"
df_meta = pd.read_csv(os.path.join(output_dir, "abag_full_dataset.csv"))
train_mask = df_meta['split'] == 'train'
test_mask = df_meta['split'] == 'test'

# Compute the random baseline PR-AUC once (equal to positive class prevalence in test set)
y_global = np.array(df_meta['label'])
random_baseline_pr_auc = y_global[test_mask].mean()

def evaluate_embeddings(file_path, model_name):
    if not os.path.exists(file_path): return None
    with open(file_path, "rb") as f: data = pickle.load(f)
    
    X, y = np.array(data["embeddings"]), np.array(data["labels"])
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    clf = LogisticRegression(max_iter=1500, n_jobs=-1, class_weight='balanced')
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    return {
        "Model": model_name,
        "Balanced Accuracy": round(balanced_accuracy_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4),
        "PR-AUC (Discovery Metric)": round(average_precision_score(y_test, y_proba), 4)
    }

models_to_eval = [
    (os.path.join(output_dir, "esm2_abag_full_results.pkl"), "ESM-2 (8M)"),
    (os.path.join(output_dir, "esm2_650m_abag_full_results.pkl"), "ESM-2 (650M)"),
    (os.path.join(output_dir, "esm2_3b_abag_full_results.pkl"), "ESM-2 (3B)"),
    (os.path.join(output_dir, "prott5_abag_full_results.pkl"), "ProtT5-XL"),
    (os.path.join(output_dir, "ankh_abag_full_results.pkl"), "Ankh-Base (Fixed)")
]

results = [res for path, name in models_to_eval if (res := evaluate_embeddings(path, name))]

if results:
    df_leaderboard = pd.DataFrame(results).sort_values(by="PR-AUC (Discovery Metric)", ascending=False)
    print(f"\n🎲 Random Baseline PR-AUC (Test Set Positive Prevalence): {round(random_baseline_pr_auc, 4)}")
    print("\n🏆 Ultimate Antibody Binding Model Leaderboard 🏆")
    print("-" * 75)
    print(df_leaderboard.to_markdown(index=False))

/home/natanon/anaconda3/envs/esm_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/natanon/anaconda3/envs/esm_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/natanon/anaconda3/envs/esm_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



🏆 Final CDR-Specific Antibody Binding Leaderboard 🏆
-------------------------------------------------------------------------------------
| Model      |   Balanced Accuracy |   ROC-AUC |   PR-AUC (Discovery Metric) |   Random Baseline PR-AUC |
|:-----------|--------------------:|----------:|----------------------------:|-------------------------:|
| ProtT5-XL  |              0.4999 |    0.8392 |                      0.1152 |                   0.0115 |
| ESM-2 (8M) |              0.5481 |    0.8468 |                      0.0343 |                   0.0115 |
| Ankh-Base  |              0.0663 |    0.0621 |                      0.0061 |                   0.0115 |


# Step 4: Output results

# ADDITIONAL: GARBAGE COLLECTION
Run this to clear GPU memory if memory allocation errors exist after running some models

In [10]:
import torch
import gc

print("--- VRAM Purge ---")

# Delete any lingering global variables if they exist in the notebook namespace
for var_name in ['model', 'tokenizer', 'embeddings', 'outputs', 'inputs']:
    if var_name in globals():
        del globals()[var_name]

# Aggressive garbage collection
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    
    device_props = torch.cuda.get_device_properties(0)
    total_vram = device_props.total_memory / 1024**3
    allocated_vram = torch.cuda.memory_allocated() / 1024**3
    reserved_vram = torch.cuda.memory_reserved() / 1024**3
    free_vram = total_vram - reserved_vram
    
    print(f"VRAM After Purge: {allocated_vram*1024:.2f} MB allocated, {reserved_vram*1024:.2f} MB reserved")
    print(f"Free VRAM available: {free_vram:.2f} GB (Total capacity: {total_vram:.2f} GB)")

--- VRAM Purge ---
VRAM After Purge: 0.00 MB allocated, 0.00 MB reserved
Free VRAM available: 22.03 GB (Total capacity: 22.03 GB)


Program run is complete: YaY! Thank you for bearing with me for the (too long) duration of execution